Checking PSD of R for DINO v1 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [ ]:
import sys
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# load model
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8').to(device)

In [ ]:
model.eval()

In [ ]:
!wget http://images.cocodataset.org/zips/test2017.zip
!unzip -q test2017.zip

In [ ]:
test_dir = "test2017"
files = os.listdir(test_dir)
len(files)

In [ ]:
def preprocessing(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_size = (224, 224)
    blob = cv2.dnn.blobFromImage(image, 1.0/255, image_size, swapRB=True, crop=False)
    blob[0][0] = (blob[0][0] - 0.485)/0.229
    blob[0][1] = (blob[0][1] - 0.456)/0.224
    blob[0][2] = (blob[0][2] - 0.406)/0.225
    return torch.tensor(blob)

In [ ]:
result = model.get_intermediate_layers(preprocessing(cv2.imread(os.path.join(test_dir, files[0]))).to(device))

In [ ]:
result[0].shape

In [ ]:
import math
math.sqrt(784)

In [ ]:
lastminW = 1e9
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    with torch.no_grad():
        blob = preprocessing(img).to(device)
        result = model.get_intermediate_layers(blob)
        feats = result[0][0][1:]
        W = torch.matmul(feats, feats.T)
        minW = W.min().item()
        d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
        mind = d.min().item()
    if minW < lastminW:
        lastminW = minW
        print(i,'W',minW)
    if mind <= 0:
        print(i,'d',mind,"!!!!!" if mind <= 0 else "")
    D = torch.diag(d.squeeze(-1))
    eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
    if eigenvalues[0] < -1e-5:
        print(i,eigenvalues[0],"!!!!!" if eigenvalues[0] <= 0 else "")
        break
    if i % 500 == 0:
        print(i,minW,mind,eigenvalues[0].item(),eigenvalues[1].item())

Though the values of W are negative, d is well-defined and R is PSD for all samples